In [1]:
# freeze_and_release_v1.R
# Mandatory freeze-and-release tasks for v1.0.0

suppressPackageStartupMessages({
  library(fs)
  library(digest)
  library(readr)
})

# ---- Settings ----
version_tag <- "1.0.0"
default_cut <- "Youden"      # pick "Youden" or "Sens>=0.80"
release_dir <- "kit/docs"
dir_create(release_dir)

# ---- Environment capture ----
r_version <- R.version.string
pkg_versions <- c(
  mirt = tryCatch(as.character(utils::packageVersion("mirt")), error = function(e) NA_character_),
  readr = tryCatch(as.character(utils::packageVersion("readr")), error = function(e) NA_character_),
  dplyr = tryCatch(as.character(utils::packageVersion("dplyr")), error = function(e) NA_character_)
)

env_lines <- c(
  sprintf("Release: %s", version_tag),
  sprintf("Date: %s", as.character(Sys.Date())),
  sprintf("R: %s", r_version),
  sprintf("mirt: %s", pkg_versions["mirt"]),
  sprintf("readr: %s", pkg_versions["readr"]),
  sprintf("dplyr: %s", pkg_versions["dplyr"])
)
writeLines(env_lines, file.path(release_dir, "environment.txt"))

# ---- Checksums for core assets ----
core_files <- c(
  "kit/assets/mod15.rds",
  "kit/assets/link_theta_to_sem.rds",
  "kit/tables/roc_operating_points.csv",
  "kit/tables/legacy_link_equiperc.csv",
  "kit/tables/legacy_link_linear_coeffs.csv",
  "kit/tables/item_parameters_min.csv"
)

sha_or_na <- function(p) if (file_exists(p)) digest(file = p, algo = "sha256") else NA_character_
sz_or_na  <- function(p) if (file_exists(p)) as.double(file_info(p)$size) else NA_real_

chk <- data.frame(
  file = core_files,
  exists = file_exists(core_files),
  size_bytes = sapply(core_files, sz_or_na),
  sha256 = sapply(core_files, sha_or_na),
  stringsAsFactors = FALSE
)
write_csv(chk, file.path(release_dir, "checksums_sha256.csv"))

# ---- Release manifest (YAML) ----
yaml_escape <- function(x) gsub(":", "\\\\:", x)
asset_yaml <- paste0(
  paste(
    sprintf("  - path: %s\n    sha256: %s\n    size_bytes: %s",
            yaml_escape(chk$file), chk$sha256, chk$size_bytes),
    collapse = "\n"
  ), "\n"
)

manifest <- c(
  sprintf("version: %s", version_tag),
  sprintf("date: %s", as.character(Sys.Date())),
  sprintf("default_cut: \"%s\"", default_cut),
  sprintf("r_version: \"%s\"", r_version),
  sprintf("mirt_version: \"%s\"", pkg_versions["mirt"]),
  "assets:",
  asset_yaml
)
writeLines(manifest, file.path(release_dir, "release_manifest.yml"))

# ---- Update example runner to pinned cut ----
runner_path <- "kit/code/run_example.R"
if (file_exists(runner_path)) {
  runner_new <- c(
    "suppressPackageStartupMessages(library(mirt))",
    "source(\"kit/code/helpers.R\")",
    "source(\"kit/code/score_shortform.R\")",
    "df <- readr::read_csv(\"kit/examples/example_input_20rows.csv\", show_col_types = FALSE)",
    sprintf("sc <- score_shortform(df, assets_dir = \"kit/assets\", tables_dir = \"kit/tables\", cut = \"%s\", legacy = TRUE, legacy_method = \"equiperc\")", default_cut),
    "readr::write_csv(sc, \"kit/examples/example_scored_output.csv\")",
    sprintf("cat(\"Scored 20-row example (cut = %s) written to kit/examples/example_scored_output.csv\\n\")", default_cut)
  )
  writeLines(runner_new, runner_path)
}

# ---- LICENSE (MIT) ----
license_txt <- sprintf(
"MIT License

Copyright (c) %s First Last

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the \"Software\"), to deal
in the Software without restriction, including without limitation the rights to
use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of
the Software, and to permit persons to whom the Software is furnished to do so,
subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED \"AS IS\", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
", format(Sys.Date(), "%Y"))
writeLines(license_txt, "kit/LICENSE.txt")

# ---- CITATION (CFF) ----
cff <- c(
  "cff-version: 1.2.0",
  sprintf("title: dasssf15: DASS 15-item Scoring with Approximate DASS-21 Display"),
  sprintf("version: %s", version_tag),
  sprintf("date-released: %s", as.character(Sys.Date())),
  "authors:",
  "  - family-names: Last",
  "    given-names: First",
  "identifiers:",
  "  - type: doi",
  "    value: 10.0000/placeholder",
  "repository-code: https://example.org/repo",
  "license: MIT",
  "message: >-",
  "  Please cite this package and kit when using the DASS-15 scorer and derived artifacts."
)
writeLines(cff, "kit/CITATION.cff")

# ---- CHANGELOG ----
chg <- c(
  sprintf("# Changelog\n\n## %s - %s", version_tag, as.character(Sys.Date())),
  "- First public release of scoring kit and dasssf15 package (assets, scorer, calibration, ROC cuts).",
  "- Pinned default ROC cut and recorded environment and checksums for reproducibility.",
  "- Added governance files (LICENSE, CITATION) and documentation (methods appendix, reproducibility guide)."
)
writeLines(chg, "kit/CHANGELOG.md")

# ---- Methods appendix ----
methods_lines <- c(
  "# Methods appendix",
  "",
  "Model and scoring:",
  "- 1D graded response model (GRM) fitted on 15 items; scoring uses mirt::fscores with method = 'EAP' and full.scores.SE for SEs. [Ref: mirt]",  # Reference in separate doc
  "- Short-form theta is linearly mapped to the SEM latent (theta_cal) via a saved linker trained on joint data. [Ref: calibration]",
  "",
  "Operating point and flag:",
  sprintf("- Elevated flag uses ROC operating points; default cut = \"%s\" (Youden) maximizing J = Sens + Spec − 1 under equal cost assumptions.", default_cut),
  "- Sensitivity-prioritized cut (e.g., Sens≥0.80) is available when false negatives are more costly.",
  "",
  "Information and reliability:",
  "- Report test information overlays vs baseline-21; marginal reliability follows rel(theta) = I(theta)/(I(theta)+1).",
  "",
  "Legacy display:",
  "- Approximate DASS-21 totals/categories via equipercentile or linear links are provided for communication only; do not use for decisions.",
  "",
  "Limitations:",
  "- Residual LD screens (e.g., Q3) are dataset dependent; thresholds are heuristic and should be contextualized.",
  "- ROC cut selection should consider prevalence and decision costs in the deployment setting."
)
writeLines(methods_lines, file.path(release_dir, "METHODS_APPENDIX.md"))

# ---- Reproducibility guide ----
repro <- c(
  "# Reproducibility guide",
  "",
  "Quick checks:",
  "- Run: Rscript kit/code/run_example.R to produce kit/examples/example_scored_output.csv.",
  "- Regenerate exhibits: source('kit/code/recreate_exhibits.R') if input CSVs are present.",
  "",
  "Environment and assets:",
  "- See kit/docs/environment.txt for R/mirt versions and kit/docs/checksums_sha256.csv for asset hashes.",
  "- The release manifest (kit/docs/release_manifest.yml) pins versions and the default ROC cut.",
  "",
  "Validation:",
  "- Use the package validator (validate_on_sample) on any dataset with dQ1S..dQ21D and the 15 short-form items to export R^2/MAE/RMSE, confusion, and decile errors."
)
writeLines(repro, file.path(release_dir, "REPRODUCIBILITY.md"))

cat("Freeze-and-release artifacts written under kit/ and kit/docs. Default cut pinned to: ", default_cut, "\n")


Freeze-and-release artifacts written under kit/ and kit/docs. Default cut pinned to:  Youden 
